# Wastewater Fits on Demonstrator

This supplementary notebook compares the fitting ensemble (simulation IDs 1-100) against the ground truth trajectory (simulation ID 0).

For each measurement station, we plot:
- median of simulations 1-100
- 50% interval (25th-75th percentile)
- 90% interval (5th-95th percentile)
- ground truth curve from simulation 0


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


In [ ]:
csv_path = "../../../preprocessing/preprocessed_data/pop8/substances/decay_Nieselregen_demonstrator_output.csv"

plot_variable = "COV19"
start_date = "2020-03-02"
save_plots = True
output_dir = "../../plots/Supplement/wastewater_fits_on_demonstrator"


In [ ]:
df = pd.read_csv(
    csv_path,
    usecols=["time_in_minutes", "value", "manhole", "simulation_id", "variable"],
)

df = df.loc[df["variable"] == plot_variable].copy()
df["manhole"] = pd.to_numeric(df["manhole"], errors="coerce").astype("Int64")
df = df.dropna(subset=["manhole"]).copy()
df["manhole"] = df["manhole"].astype(int)
df["Date"] = pd.to_datetime(start_date) + pd.to_timedelta(df["time_in_minutes"], unit="min")

print(f"Rows after filtering to {plot_variable}: {len(df):,}")
print(f"Stations: {df['manhole'].nunique()}")
print(f"Simulation IDs: {df['simulation_id'].min()}-{df['simulation_id'].max()}")


In [ ]:
fit_df = df.loc[df["simulation_id"].between(1, 100)]

fitting_summary = (
    fit_df.groupby(["manhole", "time_in_minutes"])["value"]
    .quantile([0.05, 0.25, 0.5, 0.75, 0.95])
    .unstack(level=-1)
    .rename(
        columns={
            0.05: "q05",
            0.25: "q25",
            0.50: "q50",
            0.75: "q75",
            0.95: "q95",
        }
    )
    .reset_index()
)

ground_truth = (
    df.loc[df["simulation_id"] == 0, ["manhole", "time_in_minutes", "Date", "value"]]
    .rename(columns={"value": "ground_truth"})
)

plot_df = (
    ground_truth.merge(fitting_summary, on=["manhole", "time_in_minutes"], how="inner")
    .sort_values(["manhole", "time_in_minutes"])
)

plot_df.head()


In [ ]:
if save_plots:
    os.makedirs(output_dir, exist_ok=True)

station_ids = sorted(plot_df["manhole"].unique())

for station_id in station_ids:
    station_df = plot_df.loc[plot_df["manhole"] == station_id].copy()

    fig, ax = plt.subplots(figsize=(11, 4))

    ax.fill_between(
        station_df["Date"],
        station_df["q05"],
        station_df["q95"],
        color="C0",
        alpha=0.20,
        label="90% interval",
    )
    ax.fill_between(
        station_df["Date"],
        station_df["q25"],
        station_df["q75"],
        color="C0",
        alpha=0.35,
        label="50% interval",
    )
    ax.plot(
        station_df["Date"],
        station_df["q50"],
        color="C0",
        linewidth=2.0,
        label="Median fit",
    )
    ax.plot(
        station_df["Date"],
        station_df["ground_truth"],
        color="C3",
        linewidth=1.6,
        label="Ground truth",
    )

    ax.set_title(f"Measurement station {station_id}")
    ax.set_xlabel("Date")
    ax.set_ylabel("Concentration")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="upper right", frameon=False)

    locator = mdates.AutoDateLocator()
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))

    plt.tight_layout()

    if save_plots:
        fig.savefig(
            f"{output_dir}/station_{station_id}_{plot_variable}.png",
            dpi=300,
            bbox_inches="tight",
        )

    plt.show()
